In [1]:
import numpy as np
import random
import matplotlib.pyplot as plt

In [4]:
class TicTacToe:
    def __init__(self):
        self.board = ['']*9  # A list of 9 empty strings
        self.done = False
        self.winner = None

    def reset(self):
        self.board = ['']*9
        self.done = False
        self.winner = None
        return tuple(self.board) # Returns the initial empty board state

    def available_actions(self):
        # Returns a list of indices (0-8) where the board is empty
        return [i for i, x in enumerate(self.board) if x == '']

    def check_winner(self, player):
        # Defines all 8 winning combinations (rows, cols, diagonals)
        wins = [(0,1,2), (3,4,5), (6,7,8), (0,3,6), (1,4,7), (2,5,8), (0,4,8), (2,4,6)]
        for a,b,c in wins:
            if self.board[a] == self.board[b] == self.board[c] == player:
                return True
        return False

    def step(self, action, player):
        # Check for an invalid move
        if self.board[action] != '':
            return tuple(self.board), -10, True # Penalty!

        # Place the player's mark on the board
        self.board[action] = player

        # Check for a win
        if self.check_winner(player):
            self.done = True
            self.winner = player
            return tuple(self.board), 1, True # Win reward!

        # Check for a draw (no empty spots left)
        elif '' not in self.board:
            self.done = True
            self.winner = None
            return tuple(self.board), 0.5, True # Draw reward!
        
        # Game is not over yet
        else:
            return tuple(self.board), 0, False # No reward yet

In [5]:
# Q-Learning setup
env = TicTacToe()
Q = {}             # The Q-table, stored as a dictionary
alpha = 0.1        # Learning Rate
gamma = 0.9        # Discount Factor
epsilon = 0.2      # Exploration Rate
episodes = 5000    # Number of games to play for training

def get_Q(state):
    if state not in Q:
        # If we've never seen this state, create a new row of 0s
        Q[state] = np.zeros(9)
    return Q[state]

In [6]:
# Training
for _ in range(episodes):
    state = env.reset()
    done = False
    player = 'X'
    
    while not done:
        actions = env.available_actions()
        q = get_Q(state) # Get Q-values for the current state

        # Epsilon-greedy (Explore vs. Exploit)
        if random.uniform(0, 1) < epsilon:
            action = random.choice(actions) # EXPLORE
        else:
            action = actions[np.argmax(q[actions])] # EXPLOIT

        # Take the action and get feedback from the environment
        next_state, reward, done = env.step(action, player)

        # Q-Learning Update Formula (Bellman Equation)
        q[action] = q[action] + alpha * (reward + gamma * (0 if done else np.max(get_Q(next_state))) - q[action])

        # Update state and switch player
        state = next_state
        player = 'O' if player == 'X' else 'X'

In [7]:
# Test
state = env.reset()
done = False
player = 'X'
board_states = [] # To store the game's history

while not done:
    board_states.append(state) # Save the current board
    actions = env.available_actions()
    q = get_Q(state)
    
    # Pure exploitation (no random moves)
    action = actions[np.argmax(q[actions])] 
    
    state, reward, done = env.step(action, player)
    player = 'O' if player == 'X' else 'X'

# Visualization of the game
for s in board_states:
    print(np.array(s).reshape(3,3)) # Reshape from 1x9 to 3x3
    print()

[['' '' '']
 ['' '' '']
 ['' '' '']]

[['X' '' '']
 ['' '' '']
 ['' '' '']]

[['X' 'O' '']
 ['' '' '']
 ['' '' '']]

[['X' 'O' '']
 ['' '' '']
 ['X' '' '']]

[['X' 'O' 'O']
 ['' '' '']
 ['X' '' '']]

